# RespiraNet — Hafta 3: ICBHI 2017 EDA

Bu not defteri yalnız **gerçek ICBHI dosyaları indirildikten sonra** çalıştırılmalıdır. Kaydedilmiş sahte grafik veya metrik içermez.

Önerideki hasta bazlı bölme ve dört sınıf: normal / crackle / wheeze / both.

## 1 — Yerel veri ve bölme dosyası

İlk önce repo kökünden `python -m ml.prepare_icbhi --data data/icbhi --out ml/outputs` çalıştırın.

In [ ]:
from pathlib import Path
import pandas as pd
manifest = Path("ml/outputs/cycles.csv")
assert manifest.exists(), "Önce ml.prepare_icbhi çalıştırın!"
df = pd.read_csv(manifest, dtype={"patient_id": str})
print("Hasta:", df.patient_id.nunique(), "Kayıt:", df.recording.nunique(), "Döngü:", len(df))
df.head()

## 2 — Etiket dengesi ve hasta sızıntısı denetimi

In [ ]:
import pandas as pd
pivot = pd.crosstab(df.split, df.label).reindex(index=["train","validation","test"],columns=["normal","crackle","wheeze","both"],fill_value=0)
display(pivot)
for a,b in [("train","validation"),("train","test"),("validation","test")]:
    intersection=set(df.loc[df.split==a,"patient_id"]) & set(df.loc[df.split==b,"patient_id"])
    print(a, b, "ortak hasta:",len(intersection))
    assert not intersection, "Hasta veri sızıntısı var!"

## 3 — Solunum döngüsü uzunlukları

In [ ]:
import matplotlib.pyplot as plt
ax=df.duration_s.hist(bins=35,figsize=(8,4))
ax.set(xlabel="Süre (sn)",ylabel="Döngü sayısı",title="Solunum döngüsü dağılımı")
plt.show()

## 4 — Gerçek EDA çıktısını diske kaydetme

In [ ]:
from ml.eda import run
summary = run(Path("ml/outputs/cycles.csv"), Path("ml/outputs/eda"))
summary

## 5 — Sonuç yorumu (öğrenciler tarafından doldurulacak)

- Kullanılabilen dosya/döngü sayısı ve çıkarılanlar:
- Hasta bazlı ayrımda testte kalan sınıflar:
- Veri kalitesi ve sınıf dengesizliği:
- MFCC+SVM E0 ölçümünden sonra Macro-F1 / Se / Sp:

**Not:** Resmî veri kümesindeki 920/6898 sayılarını burada ölçülmüş gibi yazmayın. Gerçek kullanım sayıları `summary.json` ile raporlanmalıdır.